# M4 robustness checks on dev (GPU)

**Question.** Does the adopted pipeline (Notebook 17 adapter, top 10 retrieved facts, top of 4 beams)
keep its answers when the input changes in ways that do not change the correct answer?

**Perturbations** (`finevidence.generation.perturbations`, all label-preserving):

- `reword`: fixed surface rewrites of the question (what is/was, percent/percentage change,
  total/overall, trailing question mark).
- `reverse_facts`: the same 10 facts in reverse order.
- `thousands_separators`: separators added to fact numbers (2457 -> 2,457; years kept).
- `foreign_fact`: the top-ranked fact of the next dev record inserted first (cannot be gold here).
- `thousands_separators_canonical`: separators added, then the pipeline's number canonicalizer
  (`finevidence.data.numbers`) applied, as for a document that arrives with separators.

`FINEVIDENCE_CONDITIONS` (comma-separated) limits the run to some conditions; `original` always runs.

Answer-changing perturbations (rescaled units, shifted periods) are out of scope: they would need new
gold labels. **Protocol.** Dev only, the frozen adapter (SHA checked) and the Notebook 10 dev rankings;
nothing is tuned. Each condition is compared with the unperturbed run on the same records: accuracy,
records fixed and broken, and an exact McNemar test. Needs a CUDA GPU; `FINEVIDENCE_SMOKE_TEST=1` runs
a tiny CPU check.

In [ ]:
from __future__ import annotations

import dataclasses
import hashlib
import json
import os
import platform
import random
import subprocess
import time
import tomllib
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import PeftModel
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoTokenizer

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.data.numbers import canonicalize_numbers
from finevidence.data.preprocessing import build_structured_input
from finevidence.generation import (
    build_retrieved_examples,
    load_rankings,
    score_program_prediction,
    summarize_scores,
    verify_rankings_file,
)
from finevidence.generation.causal import INSTRUCTION, PROGRAM_MARKER, extract_program
from finevidence.generation.inputs import format_generator_source
from finevidence.generation.perturbations import PERTURBATIONS, perturb_example

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
ADAPTER_SHA256 = "1646a66b99ae80c1b738420d94b6d589af0b1796b03c9d8cd915ea433a58b4f5"
cfg = tomllib.loads((PROJECT_ROOT / "configs" / "generator_qwen_lora_top10.toml").read_text())
assert cfg["evaluation_split"] == "dev"
pretrained = (
    "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
    if SMOKE_TEST
    else cfg["model"]["pretrained_name"]
)
num_beams = 2 if SMOKE_TEST else cfg["generation"]["num_beams"]
facts = cfg["training"]["facts_per_input"]
suffix = "-smoke" if SMOKE_TEST else ""
output_dir = PROJECT_ROOT / cfg["output"]["predictions_dir"]
KAGGLE_INPUT = Path("/kaggle/input")
found = sorted(KAGGLE_INPUT.rglob("adapter_model.safetensors")) if KAGGLE_INPUT.exists() else []
adapter_dir = Path(
    os.environ.get("FINEVIDENCE_ADAPTER_DIR")
    or (found[0].parent if found else PROJECT_ROOT / cfg["output"]["model_dir"] / "best")
)
adapter_sha = hashlib.sha256((adapter_dir / "adapter_model.safetensors").read_bytes()).hexdigest()
if not SMOKE_TEST:
    assert adapter_sha == ADAPTER_SHA256, adapter_sha

seed = cfg["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError("Needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check.")
use_fp16 = device.type == "cuda"
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "adapter_dir": str(adapter_dir),
    "adapter_sha256": adapter_sha,
    "pretrained": pretrained,
    "facts": facts,
    "num_beams": num_beams,
    "seed": seed,
}
print(json.dumps(environment, indent=2))

## 1. Dev records, rankings and perturbed inputs

In [ ]:
raw_dir = PROJECT_ROOT / cfg["dataset_root"]
print(download_finqa_splits(raw_dir, ("dev",)))
rankings_path = PROJECT_ROOT / cfg["inputs"]["rankings_path"]
verify_rankings_file(rankings_path, cfg["inputs"]["rankings_sha256"])
dev = load_finqa_split(raw_dir / "dev.json")
if SMOKE_TEST:
    dev = dev[:6]
base_examples = build_retrieved_examples(dev, load_rankings(rankings_path), facts_per_input=facts)
questions = [build_structured_input(record)["question"] for record in dev]
conditions = {"original": base_examples}
for name in PERTURBATIONS:
    conditions[name] = [
        perturb_example(
            example,
            question,
            name,
            base_examples[(index + 1) % len(base_examples)].fact_texts[0],
        )
        for index, (example, question) in enumerate(zip(base_examples, questions, strict=True))
    ]
separated = conditions["thousands_separators"]
canonical_facts = [tuple(canonicalize_numbers(f) for f in e.fact_texts) for e in separated]
conditions["thousands_separators_canonical"] = [
    dataclasses.replace(
        example, fact_texts=fact_texts, source=format_generator_source(question, fact_texts)
    )
    for example, question, fact_texts in zip(base_examples, questions, canonical_facts, strict=True)
]
wanted = os.environ.get("FINEVIDENCE_CONDITIONS")
if wanted:
    keep = {"original", *wanted.split(",")}
    conditions = {name: examples for name, examples in conditions.items() if name in keep}
changed = {
    name: sum(a.source != b.source for a, b in zip(base_examples, examples, strict=True))
    for name, examples in conditions.items()
}
print({"dev_records": len(dev), "test_loaded": False, "inputs_changed": changed})
print("first changed input:", next(iter(conditions.values()))[0].source[:200])

## 2. Decode every condition with the frozen adapter (4 beams, top beam is the answer)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(adapter_dir if not SMOKE_TEST else pretrained)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
max_source_tokens = cfg["model"]["max_source_tokens"]
marker_ids = tokenizer(PROGRAM_MARKER, add_special_tokens=False).input_ids
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
    ),
    adapter_dir,
).to(device)
model.eval()


def prompt_ids(example) -> list[int]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    return [*source[:max_source_tokens], *marker_ids]


@torch.no_grad()
def top_beam(examples) -> list[str]:
    programs = []
    batch_size = cfg["generation"]["eval_batch_size"]
    for start in range(0, len(examples), batch_size):
        prompts = [prompt_ids(example) for example in examples[start : start + batch_size]]
        width = max(len(p) for p in prompts)
        input_ids = torch.tensor(
            [[tokenizer.pad_token_id] * (width - len(p)) + p for p in prompts], device=device
        )
        attention = torch.tensor(
            [[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=device
        )
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention,
                max_new_tokens=cfg["model"]["max_new_tokens"],
                num_beams=num_beams,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        texts = tokenizer.batch_decode(generated[:, width:], skip_special_tokens=True)
        programs.extend(extract_program(text) for text in texts)
    return programs


predictions, timings = {}, {}
for name, examples in conditions.items():
    started = time.time()
    predictions[name] = top_beam(examples)
    timings[name] = round(time.time() - started, 1)
    print(f"{name}: decoded {len(examples)} records in {timings[name]} s", flush=True)
path = output_dir / f"m4-robustness{suffix}_dev_predictions.jsonl"
path.parent.mkdir(parents=True, exist_ok=True)
with path.open("w", encoding="utf-8") as handle:
    for index, record in enumerate(dev):
        row = {"record_id": str(record["id"]), "commit": commit}
        row.update({name: predictions[name][index] for name in conditions})
        handle.write(json.dumps(row) + "\n")

## 3. Accuracy under each perturbation (dev labels)

`fixed` and `broken` count records whose correctness flips relative to `original`; the p-value is an
exact two-sided McNemar test on those flips. `answer_kept` is the share of records whose program
executes to the same value as under `original`.

In [ ]:
scores = {
    name: [
        score_program_prediction(record, program, example.fact_texts)
        for record, example, program in zip(dev, conditions[name], predictions[name], strict=True)
    ]
    for name in conditions
}
base = [s.execution_correct for s in scores["original"]]
rows = []
for name in conditions:
    correct = [s.execution_correct for s in scores[name]]
    fixed = sum(c and not b for b, c in zip(base, correct, strict=True))
    broken = sum(b and not c for b, c in zip(base, correct, strict=True))
    kept = sum(
        a.valid and a.executed_value == b.executed_value
        for a, b in zip(scores["original"], scores[name], strict=True)
    )
    summary = summarize_scores(scores[name])
    rows.append(
        {
            "condition": name,
            "inputs_changed": changed[name],
            "execution_correct": summary["execution_correct"],
            "execution_accuracy": round(summary["execution_accuracy"], 4),
            "fixed": fixed,
            "broken": broken,
            "mcnemar_p": round(binomtest(fixed, fixed + broken).pvalue, 4)
            if fixed + broken
            else 1.0,
            "answer_kept": round(kept / len(dev), 4),
            "decode_seconds": timings[name],
        }
    )
report = pd.DataFrame(rows)
display(report)
(output_dir / f"m4-robustness{suffix}_dev_report.json").write_text(
    json.dumps({"environment": environment, "conditions": rows}, indent=2)
)